In [1]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *

# spark = SparkSession.builder \
#     .appName("SparkExample") \
#     .master("local[*]") \
#     .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
#     .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
#     .config('spark.executor.memory', '8g') \
#     .config('spark.driver.memory', '8g') \
#     .getOrCreate()

## actualizar retiro telef 

In [2]:
from sqlalchemy import create_engine


engine_mysql = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)

query = """
SELECT  NUMERO_DOCUMENTO,cl_telf1, cl_telf2, cl_telf3, cl_telf4, cl_telf5, cl_telf6, cl_telf7, cl_telf8, cl_telf9, cl_telf10, cl_movil, cl_celular, cl_telefono FROM crm_target.alfin_clientes
WHERE cl_base = 'mayo 2026'
and cl_estado=1
"""

df_dni = pd.read_sql(query, engine_mysql)

cols_tel = [
    'cl_telf1','cl_telf2','cl_telf3','cl_telf4','cl_telf5',
    'cl_telf6','cl_telf7','cl_telf8','cl_telf9','cl_telf10',
    'cl_movil','cl_celular','cl_telefono'
]

df_long = df_dni.melt(
    id_vars='NUMERO_DOCUMENTO',
    value_vars=cols_tel,
    var_name='tipo_telf',
    value_name='TELEFONO'
)
df_long['TELEFONO'] = (
    df_long['TELEFONO']
    .fillna(0)            
    .astype('int64')        
    .astype(str)              
)
df_long = df_long[
    (df_long['TELEFONO'].notna()) &
    (df_long['TELEFONO'] != '') &
    (df_long['TELEFONO'].str.len() == 9) &
    (df_long['TELEFONO'].str.startswith('9'))
]



In [3]:
filename='RetiroDeGestion_Telefonos.csv'
filePath = os.path.join(ruta_csv, filename)

df_list = pd.read_csv(filePath)
print(f"df_dni filas: {df_long.shape[0]}")
print(f"df_list filas: {df_list.shape[0]}")
df_list['TELEFONO'] = (
    df_list['TELEFONO']
    .fillna(0)            
    .astype('int64')        
    .astype(str)              
)

df_dni filas: 202440
df_list filas: 1327


In [ ]:

# telefonos = ["987863718", "991187143", "966700993", "997167060"]

# df_manual = pd.DataFrame(telefonos, columns=["TELEFONO"])


In [4]:
df_list = df_list.merge(
    df_long,
    on="TELEFONO",
    how="inner"
)

print(f"df_list filas: {df_list.shape[0]}")


df_list filas: 2


In [5]:
df_list['retiro']='Retirar Telef'
df_list=df_list[['NUMERO_DOCUMENTO','retiro']]
df_list.count()

NUMERO_DOCUMENTO    2
retiro              2
dtype: int64

In [6]:
update_mysql_en_bloques(
    df=df_list,
    tabla="alfin_clientes",
    periodo="mayo 2026",
    col_llave_mysql="NUMERO_DOCUMENTO",
    col_valor_mysql="estado",
    col_llave_df="NUMERO_DOCUMENTO",
    col_valor_df="retiro",
    host=server_valentina,
    user=user_valentina,
    password=pwd_valentina,
    database=db_valentina,
    port=port_mysql,
    batch_size=2000,
    validar_sin_grabar=False
)


Total registros a procesar: 2
Lote 0 - 2 actualizado | filas afectadas: 2
Proceso terminado. Total filas afectadas: 2


## actualizar retiro dni

In [7]:
from sqlalchemy import create_engine


engine_mysql = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)

query = """
SELECT 
DISTINCT 
NUMERO_DOCUMENTO as dni_cliente
FROM alfcc_clientes
WHERE cl_base = 'mayo 2026'
and cl_estado=1
"""

df_dni = pd.read_sql(query, engine_mysql)

df_dni["dni_cliente"] = (
    df_dni["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)


In [8]:
filename='RetiroDeGestion_BlackList.csv'

filePath = os.path.join(ruta_csv, filename)
df_list = pd.read_csv(filePath)

df_list.rename(columns={'DNI': 'dni_cliente'}, inplace=True)
df_list["dni_cliente"] = (
    df_list["dni_cliente"]
    .astype(str)
    .str.zfill(8)
)
print(df_list.columns)
print(df_dni.columns)


Index(['dni_cliente'], dtype='object')
Index(['dni_cliente'], dtype='object')


In [9]:
df_list = df_list.merge(
    df_dni,
    on="dni_cliente",
    how="inner"
)

print(f"df_list filas: {df_list.shape[0]}")


df_list filas: 0


In [19]:
df_list['retiro']='Retirar BlackList'
df_list=df_list[['dni_cliente','retiro']]
df_list.count()

dni_cliente    1648
retiro         1648
dtype: int64

In [ ]:
update_mysql_en_bloques(
    df=df_list,
    tabla="alfin_clientes",
    periodo="mayo 2026",
    col_llave_mysql="NUMERO_DOCUMENTO",
    col_valor_mysql="estado",
    col_llave_df="dni_cliente",
    col_valor_df="retiro",
    host=server_valentina,
    user=user_valentina,
    password=pwd_valentina,
    database=db_valentina,
    port=port_mysql,
    batch_size=2000,
    validar_sin_grabar=False
)


Total registros a procesar: 1648
Lote 0 - 1648 actualizado | filas afectadas: 0
Proceso terminado. Total filas afectadas: 0


## ejecutar query

In [ ]:
from sqlalchemy import create_engine


engine = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)


In [ ]:
with engine.begin() as conn:
    result = conn.execute(text("""
        UPDATE crm_target.alfin_clientes
        SET cl_estado = 3
        WHERE estado <> 'ACTIVO'
        AND cl_base = 'mayo 2026'
    """))
    
    print("Filas afectadas:", result.rowcount)

## actualizar dni a otro lote

In [3]:
filename='dni_repetidos_alfin.csv'

filePath = os.path.join(ruta_csv, filename)
df_list = pd.read_csv(filePath)

In [5]:
df_list["NUMERO_DOCUMENTO"] = (
    df_list["NUMERO_DOCUMENTO"]
    .astype(str)
    .str.zfill(8)
)
df_list.head()

,NUMERO_DOCUMENTO
0,46507674
1,40963267
2,41782588
3,40658062
4,09457398


In [6]:
df_list['lote_ref']='BD-Target -ASM'

In [7]:
update_mysql_en_bloques(
    df=df_list,
    tabla="alfin_clientes",
    periodo="mayo 2026",
    col_llave_mysql="NUMERO_DOCUMENTO",
    col_valor_mysql="lote",
    col_llave_df="NUMERO_DOCUMENTO",
    col_valor_df="lote_ref",
    host=server_valentina,
    user=user_valentina,
    password=pwd_valentina,
    database=db_valentina,
    port=port_mysql,
    batch_size=2000,
    validar_sin_grabar=False
)


Total registros a procesar: 1066
Lote 0 - 1066 actualizado | filas afectadas: 1066
Proceso terminado. Total filas afectadas: 1066


In [ ]:
BD-Target -ASM

In [5]:
exec_query_sql(server_zeus, "MAEBA", user_zeus, pwd_zeus, "ADM_OBJ_TG.spFunnelDinersTc", "SP funnel diners_tc Zeus")

SP funnel diners_tc Zeus | realizado | duración: 19.96 seg


In [ ]:
overwrite_table_SQL(spark,df_prueba_1,f'borrar_TARGET_202604_01',server_kishin,user_kishin,pwd_kishin,'DANTALION')
overwrite_table_SQL(spark,df_prueba_2,f'borrar_TARGET_202604_02',server_kishin,user_kishin,pwd_kishin,'DANTALION')
overwrite_table_SQL(spark,df_prueba_ch,f'borrar_TARGET_202604_ch',server_kishin,user_kishin,pwd_kishin,'DANTALION')


df_list filas: 1066


In [ ]:
df_dni filas: 128183
df_list filas: 12546

In [ ]:
ssss

In [7]:
print(df_list.columns)
print(df_long.columns)

Index(['TELEFONO'], dtype='object')
Index(['NUMERO_DOCUMENTO', 'tipo_telf', 'TELEFONO'], dtype='object')


df_list filas: 1


In [ ]:
df_list filas: 2036

NUMERO_DOCUMENTO    1
retiro              1
dtype: int64

In [10]:
df_list.head()

,NUMERO_DOCUMENTO,retiro
0,40503275,Retirar Telef


Total registros a procesar: 1
Lote 0 - 1 actualizado | filas afectadas: 1
Proceso terminado. Total filas afectadas: 1
